# Managed-memory union — layer ladder on CACTUS (dev split first)

This notebook has not been executed and contains no result.

It reuses a **completed** `architecture_retest` CACTUS run (the *prior* folder): its case roster, the memory items it already extracted and checked, and its no-RAG, raw-RAG and frozen managed answers. It only generates **new managed-arm answers**, one per layer config:

| Config | Layers |
| --- | --- |
| `L1_raw_plus_items` | raw-RAG whole messages + ranked items |
| `L2_plus_rerank` | + re-rank by recency, importance, reliability |
| `L3_plus_fact_key` | + up to 3 extra messages through the fact-key channel |
| `L4_plus_pinned_profile` | + pinned profile, v5 renderer |

Order of work: **dev split only** → read the comparison → freeze configs → prepare and run the **test** split once. Paid steps are off by default. The API key is read from the existing Colab Secret `Capstone` and never written.

In [ ]:
import os, sys, json, subprocess, importlib.util
from pathlib import Path
if not (Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG') and importlib.util.find_spec('google.colab') is not None):
    raise RuntimeError('Genuine hosted Colab required; do not connect a local runtime.')
print('hosted Colab, python', sys.version.split()[0])

In [ ]:
CODE = '56f05aa82b833b53ac57060e78abfcf2601b4b4e'  # branch feat/managed-union-layers
REPO = Path('/content/cbt-union-' + CODE[:12])
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', 'https://github.com/nanlalala/cbt-memory-system.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'feat/managed-union-layers'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', CODE], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == CODE
assert subprocess.run(['git', '-C', str(REPO), 'diff', '--quiet', 'HEAD']).returncode == 0
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'sentence-transformers'], check=True)
sys.path.insert(0, str(REPO / 'evaluation/cbt-memory-effect'))
from architecture_retest import runner, judge
from architecture_retest.transport import HostedGatewayTransport
from architecture_union import run as union_run
print('code', CODE)

## Paths
`PRIOR` must be the folder of the completed CACTUS `architecture_retest` run (it contains `protocol.json`, `cases.json` and `cases/<case_id>/`). New run folders must not exist yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PRIOR = Path('/content/drive/MyDrive/FILL_IN/prior-cactus-retest-run')   # <- set this
WORK = Path('/content/drive/MyDrive/cbt-union-runs')
WORK.mkdir(parents=True, exist_ok=True)
SPLIT = 'dev'          # change to 'test' only after configs are frozen
DEV_FAMILIES = 40
UNION = WORK / ('union-' + SPLIT + '-' + CODE[:8])
QUALITY = WORK / ('quality-' + SPLIT + '-' + CODE[:8])
assert (PRIOR / 'protocol.json').exists(), 'set PRIOR to the completed retest run folder'
print(json.dumps({'prior': str(PRIOR), 'union': str(UNION), 'quality': str(QUALITY)}, indent=1))

## 1. Prepare (no model call)
Freezes the family split, the four configs and the code hashes. Re-running this cell on an existing folder is refused.

In [ ]:
if not (UNION / 'protocol.json').exists():
    protocol = union_run.prepare(PRIOR, UNION, split=SPLIT, dev_families=DEV_FAMILIES)
else:
    protocol = json.loads((UNION / 'protocol.json').read_text())
print(json.dumps({k: protocol[k] for k in ('split', 'case_count', 'maximum_attempts', 'configs')}, indent=1))

## 2. Generate managed answers (paid, DeepSeek)
At most `case_count × 4` answer calls; identical requests share one receipt. Resumable: completed cases are skipped and a sent request is never resent automatically.

In [ ]:
from contextlib import contextmanager
@contextmanager
def api_key():
    from google.colab import userdata
    secret = userdata.get('Capstone')
    if not isinstance(secret, str) or not secret.strip():
        raise RuntimeError('Colab Secret Capstone unavailable; nothing sent.')
    try:
        yield secret
    finally:
        secret = None

RUN_ANSWERS = False
MAX_CALLS = 120
if RUN_ANSWERS:
    with api_key() as key:
        summary = union_run.run(UNION, HostedGatewayTransport(runner.IDENTITY, key), max_new_calls=MAX_CALLS, workers=4)
    print(json.dumps(summary, indent=1))
else:
    print('Paid answers disabled.', json.dumps(union_run.summarize(UNION), indent=1) if (UNION / 'cases').exists() else '')

## 3. Independent judge (paid, Qwen)
One blinded batch with the prior raw answers, the prior frozen managed answers and every config, so all conditions share one judge, prompt and session.

In [ ]:
JUDGE_IDENTITY = {'provider': 'https://tokendance.space/gateway/v1', 'model': 'qwen3.5-plus',
                  'returned_model': 'qwen3.5-plus-2026-04-20', 'model_family': 'qwen'}
if not (QUALITY / 'protocol.json').exists():
    print(union_run.export_quality(UNION, JUDGE_IDENTITY, QUALITY, maximum_attempts=2000))

RUN_JUDGE = False
JUDGE_MAX_CALLS = 200
if RUN_JUDGE:
    with api_key() as key:
        print(json.dumps(judge.run(QUALITY, HostedGatewayTransport(JUDGE_IDENTITY, key), max_new_calls=JUDGE_MAX_CALLS, workers=4), indent=1))
else:
    print('Paid judging disabled.')

## 4. Compare with raw RAG (no model call)
Paired difference per R1–R5, family-clustered bootstrap. On the dev split this is exploratory: use it to choose and freeze configs, not as the reported result.

In [ ]:
report = union_run.compare(QUALITY)
for arm, dims in report['comparisons'].items():
    print(arm)
    for d, r in dims.items():
        c = r['family_cluster_difference']
        print(f"  {d}: scored {r['scored_pairs']}/{r['planned_pairs']}  win/tie/loss {r['wins']}/{r['ties']}/{r['losses']}  mean {c['mean_difference']}  CI {c['ci95']}")

## 5. Test split (only after freezing)
Edit `union_run.DEFAULT_LADDER` in a new commit to keep only the chosen configs, set `SPLIT = 'test'` and a new `CODE`, then run sections 1–4 once. Do not tune again after seeing test results.